# Clase 189 — DoubleML / EconML: ML para causalidad

El problema: con confounding observado high-dim, regresión naive sesga. **Double/Debiased ML** (Chernozhukov 2018) usa ML para residualizar Y y T, y aplica Frisch-Waugh-Lovell sobre los residuos → estimador eficiente con CI válido.
Requiere: `pip install numpy scikit-learn` (opcional `doubleml`).

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(42)
n, p = 2000, 20
X = rng.normal(0, 1, (n, p))
beta = rng.normal(0, 1, p)
# Tratamiento depende de X (confounding)
logit = X @ beta * 0.3
T = (rng.uniform(0, 1, n) < 1/(1+np.exp(-logit))).astype(float)
# Outcome depende de X y T; ATE verdadero = 2.0
TRUE_ATE = 2.0
Y = X @ beta + TRUE_ATE * T + rng.normal(0, 1, n)
print(f'n={n}  p={p}  P(T=1)={T.mean():.3f}  TRUE ATE = {TRUE_ATE}')

## Naive: diferencia de medias (sesgado)

In [ ]:
naive = Y[T==1].mean() - Y[T==0].mean()
print(f'Naive diff-in-means = {naive:.3f}  (TRUE = {TRUE_ATE}) -> sesgado por confounding via X')

## Regresión OLS sobre X y T (no DoubleML)

In [ ]:
from numpy.linalg import lstsq
Xfull = np.column_stack([T, X, np.ones(n)])
coef, *_ = lstsq(Xfull, Y, rcond=None)
print(f'OLS coef sobre T = {coef[0]:.3f}  (mejora cuando el modelo es lineal correcto)')

## DoubleML manual — cross-fitting K=2
1. Split en K folds. En cada fold-fuera, entrenar:
   - $\hat g(X) \approx E[Y|X]$
   - $\hat m(X) \approx E[T|X]$ (propensity)
2. Residualizar: $\tilde Y = Y - \hat g(X)$, $\tilde T = T - \hat m(X)$.
3. $\hat\theta = \dfrac{\sum \tilde T \tilde Y}{\sum \tilde T^2}$ (Frisch-Waugh-Lovell).

In [ ]:
def doubleml_plr(X, T, Y, K=2, seed=42):
    n = len(Y)
    Y_res = np.zeros(n); T_res = np.zeros(n)
    kf = KFold(n_splits=K, shuffle=True, random_state=seed)
    for train_idx, test_idx in kf.split(X):
        # nuisance Y: Ridge
        g = Ridge(alpha=1.0).fit(X[train_idx], Y[train_idx])
        # nuisance T: LogReg (T binario)
        m = LogisticRegression(max_iter=1000, C=1.0).fit(X[train_idx], T[train_idx])
        Y_res[test_idx] = Y[test_idx] - g.predict(X[test_idx])
        T_res[test_idx] = T[test_idx] - m.predict_proba(X[test_idx])[:, 1]
    # FWL
    theta = (T_res * Y_res).sum() / (T_res**2).sum()
    # SE robusto (sandwich simplificado)
    psi = T_res * (Y_res - theta * T_res)
    J = (T_res**2).mean()
    var = (psi**2).mean() / (J**2) / n
    se = np.sqrt(var)
    return theta, se

theta_hat, se = doubleml_plr(X, T, Y, K=2)
ci = (theta_hat - 1.96*se, theta_hat + 1.96*se)
print(f'DoubleML manual: ATE = {theta_hat:.3f}  SE = {se:.4f}  CI95 = [{ci[0]:.3f}, {ci[1]:.3f}]')
print(f'TRUE ATE       : {TRUE_ATE}  -> dentro del CI: {ci[0] <= TRUE_ATE <= ci[1]}')

## Con más folds (K=5)

In [ ]:
for K in [2, 3, 5, 10]:
    t, s = doubleml_plr(X, T, Y, K=K)
    print(f'K={K:2d}  ATE={t:.3f}  SE={s:.4f}')

## DoubleML library (opcional)

In [ ]:
try:
    import doubleml as dml
    import pandas as pd
    df = pd.DataFrame(X, columns=[f'x{i}' for i in range(p)])
    df['Y'] = Y; df['T'] = T
    data = dml.DoubleMLData(df, y_col='Y', d_cols='T', x_cols=[f'x{i}' for i in range(p)])
    ml_g = Ridge(alpha=1.0)
    ml_m = LogisticRegression(max_iter=1000)
    plr = dml.DoubleMLPLR(data, ml_g, ml_m, n_folds=2)
    plr.fit()
    print('DoubleMLPLR:')
    print(plr.summary)
except ImportError:
    print('doubleml no instalado; usar `pip install doubleml` para validar.')
    print(f'Manual da: ATE={theta_hat:.3f}, SE={se:.4f}')

## ¿Por qué cross-fitting?
Sin cross-fitting (entrenar y predecir sobre los mismos datos), $\hat g$ y $\hat m$ **overfittean** → residuos correlacionan con T → sesgo. Cross-fitting rompe esa correlación → CI válido.

In [ ]:
# Demo: SIN cross-fitting (in-sample)
g = Ridge(alpha=1.0).fit(X, Y)
m = LogisticRegression(max_iter=1000).fit(X, T)
Y_res_is = Y - g.predict(X)
T_res_is = T - m.predict_proba(X)[:, 1]
theta_is = (T_res_is * Y_res_is).sum() / (T_res_is**2).sum()
print(f'IN-sample (sin cross-fit): ATE = {theta_is:.3f}  -> sesgado, no usar')
print(f'Cross-fitting K=5       : ATE = {doubleml_plr(X,T,Y,K=5)[0]:.3f}')

## Takeaways
1. Diff-of-means sin ajuste = **sesgado** con confounding.
2. **DoubleML** = ML flexible (Ridge/RF/XGB) + cross-fitting + FWL → ATE consistente y normal asintóticamente.
3. Cross-fitting es **crítico**: evita el sesgo por overfitting de las nuisance functions.
4. Librerías de producción: `doubleml`, `econml` (Microsoft). Soportan CATE, IV, panel.